# JuaKazi SW Bias Classifier — Fine-tune afro-xlmr-base (Kaggle)

Fine-tunes **Davlan/afro-xlmr-base** on the JuaKazi Swahili ground truth for binary gender bias classification.

**Before running:**
1. Accelerator → **GPU T4 x2** (right sidebar → Session options)
2. Add dataset: right sidebar → Input → Add input → Datasets → `juakazi-sw-ground-truth`
3. Add HF token: right sidebar → Secrets → `HF_TOKEN`

**Expected time:** ~45 min  
**Expected SW F1:** 0.88–0.93

## Step 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {gpu}  ({mem:.1f} GB VRAM)")
else:
    raise RuntimeError("No GPU — go to Session options and select T4 x2")

## Step 2 — Install dependencies

In [ ]:
%%capture
!pip install transformers>=4.40.0 accelerate huggingface_hub

## Step 3 — Load dataset

CSV is mounted automatically from the Kaggle dataset you added.

In [ ]:
import csv, os

CSV_PATH = "/kaggle/input/juakazi-sw-ground-truth/ground_truth_sw_v5.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"CSV not found at {CSV_PATH}\n"
        "Add it via: right sidebar → Input → Add input → Datasets → juakazi-sw-ground-truth"
    )

def load_dataset(path):
    texts, labels = [], []
    skipped = 0
    with open(path) as f:
        for row in csv.DictReader(f):
            text = row.get("text", "").strip()
            bias = row.get("has_bias", "").lower().strip()
            if not text or bias not in ("true", "false"):
                skipped += 1
                continue
            texts.append(text)
            labels.append(1 if bias == "true" else 0)
    print(f"Loaded {len(texts):,} rows ({skipped} skipped)")
    print(f"  BIAS=1: {sum(labels):,}  NEUTRAL=0: {len(labels)-sum(labels):,}")
    return texts, labels

texts, labels = load_dataset(CSV_PATH)


## Step 4 — Tokenize and split

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch

MODEL_ID = "Davlan/afro-xlmr-base"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

split = int(len(texts) * 0.8)
train_texts, val_texts   = texts[:split], texts[split:]
train_labels, val_labels = labels[:split], labels[split:]
print(f"Train: {len(train_texts):,}   Val: {len(val_texts):,}")

class BiasDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True, max_length=128
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = BiasDataset(train_texts, train_labels)
val_dataset   = BiasDataset(val_texts,   val_labels)
print("Datasets ready.")

In [ ]:
# Class weights — dataset is 98% NEUTRAL, 2% BIAS
# Without weighting the model learns to always predict NEUTRAL
import torch
import numpy as np

n_neg = labels.count(0)
n_pos = labels.count(1)
pos_weight = n_neg / n_pos  # ~49x
print(f'Class weight for BIAS: {pos_weight:.1f}x')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weight = torch.tensor([1.0, pos_weight], device=logits.device)
        loss = torch.nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss


## Step 5 — Load model and configure Trainer

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np

print(f"Loading model: {MODEL_ID}")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model.config.id2label = {0: "NEUTRAL", 1: "BIAS"}
model.config.label2id = {"NEUTRAL": 0, "BIAS": 1}
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="/kaggle/working/sw_bias_classifier_v1",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
print("Trainer ready.")

## Step 6 — Train

~45 min on T4 x2. Watch `eval_f1` — should reach 0.88+ by epoch 3.

In [ ]:
# Use WeightedTrainer to handle class imbalance
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training complete.")


## Step 7 — Evaluate

In [ ]:
metrics = trainer.evaluate()
print("\n=== Validation metrics ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Step 8 — Save model

In [ ]:
OUTPUT_DIR = "/kaggle/working/sw_bias_classifier_v2_final"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
) / 1e6
print(f"Saved to {OUTPUT_DIR}  ({size_mb:.0f} MB)")

## Step 9 — Push to HuggingFace Hub

Add your HF token as a Kaggle secret named `HF_TOKEN`:  
right sidebar → **Secrets** → Add new secret

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

HF_REPO = "juakazike/sw-bias-classifier-v2"
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## Done — activate in JuaKazi

Set this in your HuggingFace Space → Settings → Repository secrets:

```
JUAKAZI_ML_MODEL = juakazike/sw-bias-classifier-v2
JUAKAZI_ML_THRESHOLD = 0.75
```

The `BiasDetector` Stage 2 fallback will automatically use the fine-tuned model.